# 기본  내용 확인(모수 확인 중점)

In [7]:
"""Step 3 진단: 전체 투표 수신자, 기존 퍼널 교집합, 기간과 기록 범위.

"""

from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(frame):
        print(frame.to_string(index=False))




diag_files = {
    "accounts_user": DIAG_PROCESSED_DIR / "accounts_user_preprocessed_2.csv",
    "accounts_friendrequest": DIAG_PROCESSED_DIR / "accounts_friendrequest_preprocessed_2.csv",
    "accounts_paymenthistory": DIAG_PROCESSED_DIR / "accounts_paymenthistory_preprocessed_2.csv",
    "accounts_userquestionrecord": DIAG_PROCESSED_DIR / "accounts_userquestionrecord_preprocessed_2.csv",
    "accounts_pointhistory": DIAG_PROCESSED_DIR / "accounts_pointhistory_preprocessed_2.csv",
    "polls_questionpiece": DIAG_PROCESSED_DIR / "polls_questionpiece_preprocessed_2.csv",
    "polls_question": DIAG_INTERIM_DIR / "polls_question_preprocessed.csv",
    "polls_questionset": DIAG_INTERIM_DIR / "polls_questionset_preprocessed.csv",
    "hackle_events": DIAG_INTERIM_DIR / "hackle_events_preprocessed.csv",
    "hackle_properties": DIAG_INTERIM_DIR / "hackle_properties_preprocessed.csv",
}


def diag_show(title, frame):
    print("\n" + title)
    with pd.option_context("display.max_rows", 30, "display.max_columns", None):
        display(frame)


def diag_ids(series):
    return set(series.dropna())


def diag_pct(numerator, denominator):
    return round(numerator / denominator * 100, 2) if denominator else float("nan")


def diag_dates(series):
    # 시간대 없는 날짜는 기존 전처리 기준의 한국 시각으로 사용합니다.
    # 날짜에 시간대가 명시되어 있으면 한국 시각으로 변환합니다.
    values = pd.to_datetime(series, format="mixed", errors="coerce")
    if isinstance(values.dtype, pd.DatetimeTZDtype):
        values = values.dt.tz_convert("Asia/Seoul").dt.tz_localize(None)
    return values


# 필요한 컬럼만 읽습니다. 모든 ID를 문자열로 읽어 자료형을 맞춥니다.
# ID의 공백이나 .0 같은 표기는 자동 수정하지 않고 [5]에서 확인합니다.
diag_user = pd.read_csv(
    diag_files["accounts_user"],
    usecols=["id", "created_at"], dtype={"id": "string"},
)
diag_friend = pd.read_csv(
    diag_files["accounts_friendrequest"],
    usecols=["send_user_id", "receive_user_id", "status", "created_at"],
    dtype={"send_user_id": "string", "receive_user_id": "string", "status": "string"},
)
diag_votes = pd.read_csv(
    diag_files["accounts_userquestionrecord"],
    usecols=["id", "chosen_user_id", "created_at"],
    dtype={"id": "string", "chosen_user_id": "string"},
)

diag_user_dates = diag_dates(diag_user["created_at"])
diag_friend_dates = diag_dates(diag_friend["created_at"])
diag_vote_dates = diag_dates(diag_votes["created_at"])
diag_vote_in_period = diag_vote_dates.ge(DIAG_START) & diag_vote_dates.lt(DIAG_END)
diag_friend_in_period = diag_friend_dates.ge(DIAG_START) & diag_friend_dates.lt(DIAG_END)

# 1. 기존 퍼널을 같은 논리로 재현
# 전처리 완료 파일이므로 여기서 status, 날짜, 친구 수, 포인트 조건을 추가하지 않습니다.
diag_step1 = diag_ids(diag_user["id"])
diag_friend_users = (
    diag_ids(diag_friend["send_user_id"])
    | diag_ids(diag_friend["receive_user_id"])
)
diag_step2 = diag_step1 & diag_friend_users
diag_received_users = diag_ids(diag_votes["chosen_user_id"])
diag_step3 = diag_step2 & diag_received_users
diag_period_receivers = diag_ids(diag_votes.loc[diag_vote_in_period, "chosen_user_id"])

diag_count_result = pd.DataFrame([
    {"확인 항목": "현재 파일 전체 투표 기록 수(행)", "수": len(diag_votes)},
    {"확인 항목": "전체 고유 투표 수신자(이전 단계 무관)", "수": len(diag_received_users)},
    {"확인 항목": "공통 기간 내 고유 투표 수신자", "수": len(diag_period_receivers)},
    {"확인 항목": "기간 내 기록이 없는 수신자(날짜 미확인 포함)",
     "수": len(diag_received_users - diag_period_receivers)},
])
diag_show("[1-A] 투표 수신자 모수 확인 — 수신자 chosen_user_id 기준", diag_count_result)
print(f"파일 전체 고유 수신자: {len(diag_received_users):,}명")


diag_funnel_result = pd.DataFrame({
    "단계": ["Step 1 전체 회원", "Step 2 친구 관계 기록 보유", "Step 3 투표 수신 기록 보유"],
    "이번 계산(명)": [len(diag_step1), len(diag_step2), len(diag_step3)],
    "기존 이미지(명)": [677075, 574163, 10647],
    "직전 단계 대비(%)": [
        diag_pct(len(diag_step1), len(diag_step1)),
        diag_pct(len(diag_step2), len(diag_step1)),
        diag_pct(len(diag_step3), len(diag_step2)),
    ],
})
diag_funnel_result["기존 대비 차이(명)"] = (
    diag_funnel_result["이번 계산(명)"] - diag_funnel_result["기존 이미지(명)"]
)
diag_show("[1-B] 기존 Step 1~3 재현", diag_funnel_result)

# 2. 전체 수신자를 서로 겹치지 않는 세 집단으로 분해
diag_receivers_no_friend = (diag_received_users & diag_step1) - diag_step2
diag_receivers_no_user = diag_received_users - diag_step1
diag_partition_result = pd.DataFrame([
    {"구분": "회원 연결 + 친구 기록 있음 = Step 3", "사용자 수": len(diag_step3)},
    {"구분": "회원 연결 + 현재 친구 파일에 기록 없음", "사용자 수": len(diag_receivers_no_friend)},
    {"구분": "현재 회원 파일과 ID 미연결", "사용자 수": len(diag_receivers_no_user)},
])
diag_partition_result["전체 수신자 대비(%)"] = diag_partition_result["사용자 수"].map(
    lambda n: diag_pct(n, len(diag_received_users))
)
diag_show("[2] 전체 수신자와 Step 3의 차이 — 아래 세 집단의 합 = 전체 수신자", diag_partition_result)
print(f"Step 2 중 수신 기록이 없는 사용자: {len(diag_step2 - diag_received_users):,}명")
print("친구 파일에 기록이 없다는 것은 과거에도 친구가 없었다는 뜻은 아닙니다.")
print("수신 기록이 없는 사용자를 이탈자로 확정하지 않습니다.")

# 3. 실제 기록 범위와 날짜 상태: 회원은 시작일 이전 가입자도 정상적으로 포함
diag_period_rows = []
for name, frame, dates in [
    ("accounts_user", diag_user, diag_user_dates),
    ("accounts_friendrequest", diag_friend, diag_friend_dates),
    ("accounts_userquestionrecord", diag_votes, diag_vote_dates),
]:
    diag_period_rows.append({
        "테이블": name,
        "전체 행": len(frame),
        "최초 created_at": dates.min(),
        "최종 created_at": dates.max(),
        "2023-05-13 이전 행": int(dates.lt(DIAG_START).sum()),
        "공통 기간 내 행": int((dates.ge(DIAG_START) & dates.lt(DIAG_END)).sum()),
        "2024-05-08 이후 행": int(dates.ge(DIAG_END).sum()),
        "원래 날짜 결측 행": int(frame["created_at"].isna().sum()),
        "날짜 해석 실패 행": int((frame["created_at"].notna() & dates.isna()).sum()),
    })
diag_period_result = pd.DataFrame(diag_period_rows)
diag_show("[3] 기간 점검 — 종료 기준은 2024-05-08 00:00 미만",
          diag_period_result.set_index("테이블").T)
print("accounts_user의 2023-05-13 이전 가입자는 포함하는 것이 기존 기준입니다.")
print("앞 단계 조건은 그대로 두고 기간 내 수신 기록만 사용한 Step 3:",
      f"{len(diag_step2 & diag_period_receivers):,}명")

# 4. 월별 기록 분포: 기록이 전혀 없는 월도 0으로 출력
diag_months = pd.period_range(DIAG_START, DIAG_END - pd.Timedelta(days=1), freq="M")
diag_vote_monthly_source = pd.DataFrame({
    "월": diag_vote_dates.loc[diag_vote_in_period].dt.to_period("M"),
    "기록일": diag_vote_dates.loc[diag_vote_in_period].dt.normalize(),
    "수신자": diag_votes.loc[diag_vote_in_period, "chosen_user_id"],
})
diag_vote_monthly_source["Step2에_속한_수신자"] = diag_vote_monthly_source["수신자"].where(
    diag_vote_monthly_source["수신자"].isin(diag_step2)
)
diag_monthly_result = diag_vote_monthly_source.groupby("월").agg(
    투표기록수=("수신자", "size"),
    고유수신자수=("수신자", "nunique"),
    Step2내_고유수신자수=("Step2에_속한_수신자", "nunique"),
    투표기록이_있는_일수=("기록일", "nunique"),
).reindex(diag_months, fill_value=0)
diag_monthly_result["친구요청_기록수"] = (
    diag_friend_dates.loc[diag_friend_in_period].dt.to_period("M").value_counts()
    .reindex(diag_months, fill_value=0)
)
diag_monthly_result["분석범위_달력일수"] = (
    pd.Series(pd.date_range(DIAG_START, DIAG_END - pd.Timedelta(days=1)))
    .dt.to_period("M").value_counts().reindex(diag_months, fill_value=0)
)
diag_monthly_result.index.name = "월"
diag_monthly_result = diag_monthly_result.reset_index()
diag_show("[4] 월별 기록 분포", diag_monthly_result)
print("2023년 5월은 13일부터, 2024년 5월은 7일까지입니다.")
print("같은 사람이 여러 달에 등장하므로 월별 고유 수신자 수를 더하면 중복됩니다.")
print("기록 없는 날은 실제 활동 부재와 데이터 누락 중 어느 쪽인지 추가 확인이 필요합니다.")

# 5. ID 상태, 중복, 친구 상태 코드 확인
diag_id_rows = []
for label, values in [
    ("accounts_user.id", diag_user["id"]),
    ("friendrequest.send_user_id", diag_friend["send_user_id"]), 
    ("friendrequest.receive_user_id", diag_friend["receive_user_id"]),
    ("userquestionrecord.chosen_user_id", diag_votes["chosen_user_id"]),
]:
    diag_id_rows.append({
        "ID 컬럼": label,
        "결측 행": int(values.isna().sum()),
        "공백만 있는 행": int(values.str.fullmatch(r"\s*").fillna(False).sum()),
        "앞뒤 공백이 있는 행": int(values.ne(values.str.strip()).fillna(False).sum()),
        "123.0 형태인 행": int(values.str.fullmatch(r"[0-9]+\.0+").fillna(False).sum()),
        "고유 ID 수": values.nunique(),
    })
diag_id_result = pd.DataFrame(diag_id_rows)
diag_show("[5-A] 사용자 ID 점검 — 문자열로 읽되 표기 자체는 유지", diag_id_result)
if diag_id_result[["공백만 있는 행", "앞뒤 공백이 있는 행", "123.0 형태인 행"]].to_numpy().sum():
    print("ID 표기 차이가 있습니다. [1-B]의 인원 차이나 [2]의 미연결이 이 영향인지 확인해야 합니다.")
diag_show("[5-B] 중복 ID 점검", pd.DataFrame([
    {"확인 항목": "회원 id 중복 초과 행", "수": int(diag_user["id"].dropna().duplicated().sum())},
    {"확인 항목": "투표 기록 id 결측 행", "수": int(diag_votes["id"].isna().sum())},
    {"확인 항목": "투표 기록 id 중복 초과 행", "수": int(diag_votes["id"].dropna().duplicated().sum())},
]))






[1-A] 투표 수신자 모수 확인 — 수신자 chosen_user_id 기준


,확인 항목,수
0,현재 파일 전체 투표 기록 수(행),800793
1,전체 고유 투표 수신자(이전 단계 무관),13811
2,공통 기간 내 고유 투표 수신자,13811
3,기간 내 기록이 없는 수신자(날짜 미확인 포함),0


파일 전체 고유 수신자: 13,811명

[1-B] 기존 Step 1~3 재현


,단계,이번 계산(명),기존 이미지(명),직전 단계 대비(%),기존 대비 차이(명)
0,Step 1 전체 회원,677075,677075,100.00,0
1,Step 2 친구 관계 기록 보유,574163,574163,84.80,0
2,Step 3 투표 수신 기록 보유,10647,10647,1.85,0



[2] 전체 수신자와 Step 3의 차이 — 아래 세 집단의 합 = 전체 수신자


...

,월,투표기록수,고유수신자수,Step2내_고유수신자수,투표기록이_있는_일수,친구요청_기록수,분석범위_달력일수
0,2023-05,718718,12882,9915,19,7390596,19
1,2023-06,71341,7932,6307,30,366848,30
2,2023-07,5396,2433,1827,31,28323,31
3,2023-08,2050,1135,963,31,3122,31
4,2023-09,1665,1078,865,28,2214,30
5,2023-10,330,244,208,27,571,31
6,2023-11,253,153,138,17,658,30
7,2023-12,502,209,188,24,197,31
8,2024-01,236,149,120,20,230,31
9,2024-02,154,92,72,12,42,29


2023년 5월은 13일부터, 2024년 5월은 7일까지입니다.
같은 사람이 여러 달에 등장하므로 월별 고유 수신자 수를 더하면 중복됩니다.
기록 없는 날은 실제 활동 부재와 데이터 누락 중 어느 쪽인지 추가 확인이 필요합니다.

[5-A] 사용자 ID 점검 — 문자열로 읽되 표기 자체는 유지


,ID 컬럼,결측 행,공백만 있는 행,앞뒤 공백이 있는 행,123.0 형태인 행,고유 ID 수
0,accounts_user.id,0,0,0,0,677075
1,friendrequest.send_user_id,0,0,0,0,475793
2,friendrequest.receive_user_id,0,0,0,0,508356
3,userquestionrecord.chosen_user_id,0,0,0,0,13811



[5-B] 중복 ID 점검


,확인 항목,수
0,회원 id 중복 초과 행,0
1,투표 기록 id 결측 행,0
2,투표 기록 id 중복 초과 행,0


# 추가 분석 내용

In [ ]:
"""Step 3 후속 확인: 과거 친구 기록 → 월별 비교 → 동일 관찰 기간.


A: 현재 Step 3 밖의 수신자가 기간 이전 친구 기록을 보유하는지 확인합니다.
B: 같은 달 친구 기록 사용자 / 요청 발신자 중 수신 기록 보유 비율을 비교합니다.
C: 기간 내 첫 수락 상태 요청 발신 이후 7일·30일 수신 비율을 비교합니다.
   7일·30일은 확인용 설정이며, 두 비율의 분모는 30일 관찰 가능한 동일 집단입니다.

created_at은 요청 생성 시각입니다. 친구 수락 완료 시각으로 사용하지 않습니다.
이 결과는 기존 Step 3의 1.85%와 정의가 다른 추가 비교입니다.
"""

import pandas as pd

NEXT_WINDOWS = [7, 30]
NEXT_MAX_DAYS = max(NEXT_WINDOWS)
NEXT_OBSERVATION_END = DIAG_END  # 2024-05-08 00:00 미포함

# ------------------------------------------------------------
# A. Step 3 밖 수신자의 기간 이전 친구 기록 확인
# ------------------------------------------------------------
next_missing_friend = diag_receivers_no_friend
next_old_friend_path = DIAG_INTERIM_DIR / "accounts_friendrequest_preprocessed.csv"
next_history_result = None
next_population_result = None

if next_old_friend_path.is_file():
    next_old_before_ids = set()
    next_old_period_ids = set()
    next_old_date_missing = 0
    next_old_date_failed = 0
    next_old_rows = 0
    next_old_before_rows = 0
    next_old_period_rows = 0
    next_old_min = pd.NaT
    next_old_max = pd.NaT

    # 이전 친구 파일은 크므로 30만 행씩 읽어 사용자 집합만 합칩니다.
    for next_chunk in pd.read_csv(
        next_old_friend_path,
        usecols=["send_user_id", "receive_user_id", "status", "created_at"],
        dtype={"send_user_id": "string", "receive_user_id": "string", "status": "string"},
        chunksize=300000,
    ):
        next_dates = diag_dates(next_chunk["created_at"])
        next_is_accepted = next_chunk["status"].eq("A").fillna(False)
        next_before = next_is_accepted & next_dates.lt(DIAG_START)
        next_during = next_is_accepted & next_dates.ge(DIAG_START) & next_dates.lt(DIAG_END)

        next_old_rows += len(next_chunk)
        next_old_date_missing += int(next_chunk["created_at"].isna().sum())
        next_old_date_failed += int((next_chunk["created_at"].notna() & next_dates.isna()).sum())
        next_old_before_rows += int(next_before.sum())
        next_old_period_rows += int(next_during.sum())
        if next_dates.notna().any():
            next_min = next_dates.min()
            next_max = next_dates.max()
            next_old_min = next_min if pd.isna(next_old_min) else min(next_old_min, next_min)
            next_old_max = next_max if pd.isna(next_old_max) else max(next_old_max, next_max)

        for next_col in ["send_user_id", "receive_user_id"]:
            next_old_before_ids.update(next_chunk.loc[next_before, next_col].dropna())
            next_old_period_ids.update(next_chunk.loc[next_during, next_col].dropna())

    # 아래 세 집단은 겹치지 않으며, 합계는 현재 Step 3 밖의 수신자 수입니다.
    next_old_during_only = next_missing_friend & next_old_period_ids
    next_old_before_only = (next_missing_friend - next_old_during_only) & next_old_before_ids
    next_old_unconfirmed = next_missing_friend - next_old_during_only - next_old_before_only
    next_history_result = pd.DataFrame([
        {"구분": "이전 파일의 공통 기간 내 A 기록 있음", "사용자 수": len(next_old_during_only)},
        {"구분": "공통 기간 내 A 기록 없고, 2023-05-13 이전 A 기록 있음", "사용자 수": len(next_old_before_only)},
        {"구분": "이전 파일에서도 종료일 이전 A 기록 확인 안 됨", "사용자 수": len(next_old_unconfirmed)},
    ])
    diag_show("[A-1] 현재 Step 3 밖 수신자의 과거 친구 기록", next_history_result)
    print(f"분류 대상: {len(next_missing_friend):,}명 / 세 집단 합: {next_history_result['사용자 수'].sum():,}명")
    print("이전 파일:", next_old_friend_path)
    print(f"전체 {next_old_rows:,}행 / 최초 {next_old_min} / 최종 {next_old_max}")
    print(f"기간 이전 A 기록 {next_old_before_rows:,}행 / 공통 기간 내 A 기록 {next_old_period_rows:,}행")
    print(f"날짜 결측 {next_old_date_missing:,}행 / 날짜 해석 실패 {next_old_date_failed:,}행")
    print("첫 집단이 있으면 기간 제한 이외의 필터·파일 버전 차이도 확인해야 합니다.")
    print("마지막 집단은 제공된 이전 파일에서도 확인되지 않는다는 뜻입니다.")

    # 과거 A 기록을 포함하면 분자뿐 아니라 분모도 함께 바꿔 비교합니다.
    next_with_history = diag_step1 & (diag_friend_users | next_old_before_ids)
    next_history_receivers = next_with_history & diag_received_users
    next_population_result = pd.DataFrame([
        {"기준": "기존: 공통 기간 내 A 요청 기록 사용자", "분모(명)": len(diag_step2),
         "그중 기간 내 수신자(명)": len(diag_step3),
         "기록 보유 비율(%)": diag_pct(len(diag_step3), len(diag_step2))},
        {"기준": "비교: 위 집단 + 기간 이전 A 요청 기록 사용자", "분모(명)": len(next_with_history),
         "그중 기간 내 수신자(명)": len(next_history_receivers),
         "기록 보유 비율(%)": diag_pct(len(next_history_receivers), len(next_with_history))},
    ])
    diag_show("[A-2] 과거 기록 포함 여부에 따른 모집단 비교", next_population_result)
    print("추가된 사용자가 현재도 친구 관계를 유지한다는 의미는 아닙니다.")
else:
    print("[A] 이전 친구 파일이 없어 과거 기록 확인을 생략합니다:", next_old_friend_path)

# ------------------------------------------------------------
# B. 월별 분모 비교 — 수신 여부를 분모 선정 조건으로 쓰지 않습니다.
# ------------------------------------------------------------
next_months = pd.period_range(DIAG_START, DIAG_END - pd.Timedelta(days=1), freq="M")
next_month_rows = []

for next_month in next_months:
    next_start = max(DIAG_START, next_month.start_time)
    next_end = min(DIAG_END, (next_month + 1).start_time)
    next_f_mask = diag_friend_dates.ge(next_start) & diag_friend_dates.lt(next_end)
    next_v_mask = diag_vote_dates.ge(next_start) & diag_vote_dates.lt(next_end)

    next_senders = set(diag_friend.loc[next_f_mask, "send_user_id"].dropna()) & diag_step1
    next_targets = set(diag_friend.loc[next_f_mask, "receive_user_id"].dropna()) & diag_step1
    next_friend_people = next_senders | next_targets
    next_receivers = set(diag_votes.loc[next_v_mask, "chosen_user_id"].dropna()) & diag_step1

    next_month_rows.append({
        "월": str(next_month),
        "친구기록 사용자(명)": len(next_friend_people),
        "그중 수신자(명)": len(next_friend_people & next_receivers),
        "친구기록 기준 비율(%)": diag_pct(len(next_friend_people & next_receivers), len(next_friend_people)),
        "요청 발신자(명)": len(next_senders),
        "발신자 중 수신자(명)": len(next_senders & next_receivers),
        "발신자 기준 비율(%)": diag_pct(len(next_senders & next_receivers), len(next_senders)),
    })

next_month_result = pd.DataFrame(next_month_rows)
diag_show("[B] 같은 달의 친구 기록·발신자 기준 수신 기록 보유 비율", next_month_result)
print("친구 기록은 전처리된 A 상태 요청이며, 발신자와 수신자를 모두 포함합니다.")
print("요청 발신자는 실제 발신 행동이 확인되는 집단이지만, 앱의 모든 활동 사용자를 대표하지는 않습니다.")
print("월별 비율은 같은 달 두 기록의 동시 보유 비율입니다. 발생 순서는 반영하지 않았습니다.")
print("첫 달은 2023-05-13부터, 마지막 달은 2024-05-07까지입니다. 월별 고유 사용자 수는 합산하지 않습니다.")

# ------------------------------------------------------------
# C. 요청 발신 이후 같은 길이의 관찰 기간 적용
#    친구 수락 시점은 확인되지 않았으므로 실제 요청 발신 시점을 사용합니다.
# ------------------------------------------------------------
next_friend_period = diag_friend_dates.ge(DIAG_START) & diag_friend_dates.lt(DIAG_END)
next_first_send = (
    diag_friend_dates.loc[next_friend_period]
    .groupby(diag_friend.loc[next_friend_period, "send_user_id"])
    .min()
    .rename("request_at")
    .rename_axis("user_id")
    .reset_index()
)
next_first_send = next_first_send.loc[next_first_send["user_id"].isin(diag_step1)].assign(
    user_id=lambda frame: frame["user_id"].astype("string")
)

next_vote_period = diag_vote_dates.ge(DIAG_START) & diag_vote_dates.lt(DIAG_END)
next_receive_events = pd.DataFrame({
    "user_id": diag_votes.loc[next_vote_period, "chosen_user_id"].astype("string"),
    "received_at": diag_vote_dates.loc[next_vote_period],
}).dropna(subset=["user_id", "received_at"])

# 사용자마다 발신 '이후' 첫 수신을 찾습니다.
# 사용자 전체의 최초 수신을 먼저 뽑으면 발신 뒤에 발생한 수신을 놓칠 수 있습니다.
# merge_asof를 사용해 사용자별 모든 기록 조합을 만드는 대용량 조인을 피합니다.
next_timed_users = pd.merge_asof(
    next_first_send.sort_values("request_at"),
    next_receive_events.sort_values("received_at"),
    left_on="request_at", right_on="received_at", by="user_id",
    direction="forward", allow_exact_matches=False,
)

# 모든 비교 기간을 관찰할 수 있는 동일 집단만 분모에 넣습니다.
# 종료일은 미포함이고, 각 관찰 기한의 정확한 시각은 포함하므로 < 조건을 사용합니다.
next_timed_users["observable"] = (
    next_timed_users["request_at"] + pd.Timedelta(days=NEXT_MAX_DAYS)
).lt(NEXT_OBSERVATION_END)
next_timed_users["월"] = next_timed_users["request_at"].dt.to_period("M")

next_observed_count = int(next_timed_users["observable"].sum())
next_window_rows = []
for next_days in NEXT_WINDOWS:
    next_flag = f"received_{next_days}d"
    next_timed_users[next_flag] = (
        next_timed_users["observable"]
        & next_timed_users["received_at"].notna()
        & next_timed_users["received_at"].le(
            next_timed_users["request_at"] + pd.Timedelta(days=next_days)
        )
    )
    next_received_count = int(next_timed_users[next_flag].sum())
    next_window_rows.append({
        "발신 이후 관찰 기간": f"{next_days}일",
        "공통 분모(명)": next_observed_count,
        "기한 내 후속 수신자(명)": next_received_count,
        "후속 수신 비율(%)": diag_pct(next_received_count, next_observed_count),
    })

next_window_result = pd.DataFrame(next_window_rows)
diag_show("[C-1] 기간 내 첫 A 상태 요청 발신 이후 수신 비율", next_window_result)
print(f"기간 내 요청 발신자: {len(next_timed_users):,}명")
print(f"{NEXT_MAX_DAYS}일 관찰 가능한 공통 분모: {next_observed_count:,}명")
print(f"관찰 시간이 부족해 비교에서 제외: {len(next_timed_users) - next_observed_count:,}명")
print("같은 시각의 수신은 제외하고, 발신 이후 정확히 7일·30일인 수신은 해당 기한에 포함합니다.")
print("생애 첫 수신 여부는 확인하지 않습니다. 발신 이후 추가 수신이 있는지를 봅니다.")

next_aggregations = {
    "기간내첫발신자(명)": ("user_id", "size"),
    "공통관찰가능자(명)": ("observable", "sum"),
}
for next_days in NEXT_WINDOWS:
    next_aggregations[f"{next_days}일내수신자(명)"] = (f"received_{next_days}d", "sum")

next_cohort_result = (
    next_timed_users.groupby("월").agg(**next_aggregations)
    .reindex(next_months, fill_value=0)
)
next_cohort_result["관찰부족(명)"] = (
    next_cohort_result["기간내첫발신자(명)"] - next_cohort_result["공통관찰가능자(명)"]
)
for next_days in NEXT_WINDOWS:
    next_cohort_result[f"{next_days}일내수신율(%)"] = (
        next_cohort_result[f"{next_days}일내수신자(명)"]
        .div(next_cohort_result["공통관찰가능자(명)"].replace(0, float("nan")))
        .mul(100).round(2)
    )
next_cohort_result.index.name = "기간내첫발신월"
next_cohort_result = next_cohort_result.reset_index()
diag_show("[C-2] 발신 시기별 비교 — 관찰 가능한 사용자가 없으면 비율은 NaN", next_cohort_result)
print("관찰 기간 비교는 종료일까지 기록이 충분히 수집되어 있다는 전제입니다.")
print("기록의 보존·추출 범위가 불완전한 문제는 관찰 기간을 맞추는 것만으로 해결되지 않습니다.")
print("\n[A-1], [A-2], [B], [C-1], [C-2] 결과를 확인한 뒤 팀의 최종 분석 기준을 정하세요.")



[A-1] 현재 Step 3 밖 수신자의 과거 친구 기록


,구분,사용자 수
0,이전 파일의 공통 기간 내 A 기록 있음,0
1,"공통 기간 내 A 기록 없고, 2023-05-13 이전 A 기록 있음",2985
2,이전 파일에서도 종료일 이전 A 기록 확인 안 됨,179


분류 대상: 3,164명 / 세 집단 합: 3,164명
이전 파일: /Users/apple/Documents/team4-final-project/data/interim/accounts_friendrequest_preprocessed.csv
전체 17,147,175행 / 최초 2023-04-17 18:29:11 / 최종 2024-05-09 09:21:47
기간 이전 A 기록 5,085,264행 / 공통 기간 내 A 기록 7,793,140행
날짜 결측 0행 / 날짜 해석 실패 0행
첫 집단이 있으면 기간 제한 이외의 필터·파일 버전 차이도 확인해야 합니다.
마지막 집단은 제공된 이전 파일에서도 확인되지 않는다는 뜻입니다.

[A-2] 과거 기록 포함 여부에 따른 모집단 비교


,기준,분모(명),그중 기간 내 수신자(명),기록 보유 비율(%)
0,기존: 공통 기간 내 A 요청 기록 사용자,574163,10647,1.85
1,비교: 위 집단 + 기간 이전 A 요청 기록 사용자,650275,13632,2.10


추가된 사용자가 현재도 친구 관계를 유지한다는 의미는 아닙니다.

[B] 같은 달의 친구 기록·발신자 기준 수신 기록 보유 비율


...

친구 기록은 전처리된 A 상태 요청이며, 발신자와 수신자를 모두 포함합니다.
요청 발신자는 실제 발신 행동이 확인되는 집단이지만, 앱의 모든 활동 사용자를 대표하지는 않습니다.
월별 비율은 같은 달 두 기록의 동시 보유 비율입니다. 발생 순서는 반영하지 않았습니다.
첫 달은 2023-05-13부터, 마지막 달은 2024-05-07까지입니다. 월별 고유 사용자 수는 합산하지 않습니다.

[C-1] 기간 내 첫 A 상태 요청 발신 이후 수신 비율


,발신 이후 관찰 기간,공통 분모(명),기한 내 후속 수신자(명),후속 수신 비율(%)
0,7일,475766,6277,1.32
1,30일,475766,7417,1.56


기간 내 요청 발신자: 475,793명
30일 관찰 가능한 공통 분모: 475,766명
관찰 시간이 부족해 비교에서 제외: 27명
같은 시각의 수신은 제외하고, 발신 이후 정확히 7일·30일인 수신은 해당 기한에 포함합니다.
생애 첫 수신 여부는 확인하지 않습니다. 발신 이후 추가 수신이 있는지를 봅니다.

[C-2] 발신 시기별 비교 — 관찰 가능한 사용자가 없으면 비율은 NaN


,기간내첫발신월,기간내첫발신자(명),공통관찰가능자(명),7일내수신자(명),30일내수신자(명),관찰부족(명),7일내수신율(%),30일내수신율(%)
0,2023-05,452254,452254,6193,7319,0,1.37,1.62
1,2023-06,19806,19806,83,94,0,0.42,0.47
2,2023-07,2138,2138,1,4,0,0.05,0.19
3,2023-08,463,463,0,0,0,0.00,0.00
4,2023-09,434,434,0,0,0,0.00,0.00
5,2023-10,151,151,0,0,0,0.00,0.00
6,2023-11,244,244,0,0,0,0.00,0.00
7,2023-12,73,73,0,0,0,0.00,0.00
8,2024-01,97,97,0,0,0,0.00,0.00
9,2024-02,24,24,0,0,0,0.00,0.00


관찰 기간 비교는 종료일까지 기록이 충분히 수집되어 있다는 전제입니다.
기록의 보존·추출 범위가 불완전한 문제는 관찰 기간을 맞추는 것만으로 해결되지 않습니다.

[A-1], [A-2], [B], [C-1], [C-2] 결과를 확인한 뒤 팀의 최종 분석 기준을 정하세요.
